In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import joblib
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
from sklearn.metrics import classification_report
from tqdm import tqdm
import time


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/bert-text-phishing-model-i/pytorch/default/1/config.json
/kaggle/input/bert-text-phishing-model-i/pytorch/default/1/tokenizer.json
/kaggle/input/bert-text-phishing-model-i/pytorch/default/1/tokenizer_config.json
/kaggle/input/bert-text-phishing-model-i/pytorch/default/1/model.safetensors
/kaggle/input/bert-text-phishing-model-i/pytorch/default/1/special_tokens_map.json
/kaggle/input/bert-text-phishing-model-i/pytorch/default/1/vocab.txt
/kaggle/input/bert-text-phishing-model-ii/pytorch/default/1/config.json
/kaggle/input/bert-text-phishing-model-ii/pytorch/default/1/tokenizer.json
/kaggle/input/bert-text-phishing-model-ii/pytorch/default/1/tokenizer_config.json
/kaggle/input/bert-text-phishing-model-ii/pytorch/default/1/model.safetensors
/kaggle/input/bert-text-phishing-model-ii/pytorch/default/1/special_tokens_map.json
/kaggle/input/bert-text-phishing-model-ii/pytorch/default/1/vocab.txt
/kaggle/input/url-pishing-xgboost-statistical-model/scikitlearn/default/1/url_xgboos

In [2]:
TEST_DATA_PATH = "/kaggle/input/baitblock-joint-test-data/BaitBlockJointModelTestDatasetWithURLFeatures.csv"
BERT1_MODEL_DIR = "/kaggle/input/bert-text-phishing-model-i/pytorch/default/1"
BERT2_MODEL_DIR = "/kaggle/input/bert-text-phishing-model-ii/pytorch/default/1"
XGBOOST_URL_MODEL_PATH = "/kaggle/input/url-pishing-xgboost-statistical-model/scikitlearn/default/1/url_xgboost.pkl"

In [3]:
with open(XGBOOST_URL_MODEL_PATH, 'rb') as file:
    url_model = joblib.load(file)
print(type(url_model))

<class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [4]:
text_model1 = AutoModelForSequenceClassification.from_pretrained(BERT1_MODEL_DIR)
text_model2 = AutoModelForSequenceClassification.from_pretrained(BERT2_MODEL_DIR)
tokenizer1 = AutoTokenizer.from_pretrained(BERT1_MODEL_DIR)
tokenizer2 = AutoTokenizer.from_pretrained(BERT2_MODEL_DIR)
print(type(text_model1))

<class 'transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification'>


In [6]:
def extract_url_features(row):
    feature_names = ['TLDLegitimateProb', 'NoOfURLRedirect', 'IsDomainIP_1', 'HasObfuscation_1', 'IsHTTPS_1','HasExternalFormSubmit_1','URL_Domain_Interaction']
    features = row[feature_names].values.reshape(1, -1)  # Get the feature values
    features_df = pd.DataFrame(features, columns=feature_names)
    return features_df

In [7]:
def extract_text_features(row):
    return row['text']

In [13]:
def combined_prediction(row, feature_extractors, models, weights, tokenizer=None):
    phishing_score = 0
    weights_sum = 0
    
    for extract_features, model, weight in zip(feature_extractors, models, weights):
        features = extract_features(row)
        sub_prediction = 0
        module_name = model.__class__.__module__
        # For scikit-learn models
        if 'sklearn' in module_name:
            sub_prediction = (1 - model.predict(features)) * weight
#             print("URL",sub_prediction," | ",end="")
        # For Hugging Face transformers models
        elif 'transformers.models' in module_name:
            if tokenizer is not None:
                # Tokenize the text feature
                tokenized_input = tokenizer(features, return_tensors='pt', padding=True, truncation=True)
                with torch.no_grad():
                    output = model(**tokenized_input)
                logits = output.logits
                probabilities = torch.softmax(logits, dim=-1)
                sub_prediction = probabilities[0][1].item() * weight
#                 print("TEXT",sub_prediction," | ",end="")
        phishing_score += sub_prediction
        weights_sum += weight
    prediction = phishing_score / weights_sum
    return prediction

In [9]:
def evaluate_model(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_pred_binary = (y_pred > 0.5).astype(int)
    report = classification_report(y_true, y_pred_binary)
    print('Classification Report:')
    print(report)

In [14]:
def test(
    test_df,
    true_labels,
    text_model, 
    tokenizer,
    text_extractor=extract_text_features, 
    text_weight=0.5, 
    url_model=url_model, 
    url_extractor=extract_url_features, 
    url_weight=0.5):
    
    predictions = []
    total_rows = len(test_df)
    start_time = time.time()

    # Initialize the progress bar
    with tqdm(total=total_rows, desc="Processing rows", unit="row") as pbar:
        for index, row in test_df.iterrows():
            features_extractors = [text_extractor]
            models = [text_model]
            weights = [text_weight]
            is_url = row['isURL']
            if is_url == 1:
                features_extractors.append(url_extractor)
                models.append(url_model)
                weights.append(url_weight)
                
            prediction = combined_prediction(row, features_extractors, models, weights, tokenizer)
            if isinstance(prediction, np.ndarray):
                prediction = float(prediction[0])
#             print("SCPRE",prediction," | LABEL",true_labels.iloc[index])
            predictions.append(prediction)
        
            # Update the progress bar
            pbar.update(1)
            
            # Calculate time remaining
            elapsed_time = time.time() - start_time
            average_time_per_row = elapsed_time / (index + 1)
            remaining_rows = total_rows - (index + 1)
            estimated_time_remaining = int(average_time_per_row * remaining_rows)
            
            # Update progress bar description with time remaining
            pbar.set_postfix({
                "Time Remaining": f"{estimated_time_remaining//60}min{estimated_time_remaining%60}s"
            })

    evaluate_model(true_labels, predictions)

In [11]:
test_df = pd.read_csv(TEST_DATA_PATH)
labels = test_df['label']
test_raw_df = test_df.drop(columns=['label'])

In [11]:
test(test_raw_df, labels, text_model1, tokenizer1,text_weight=0.7,url_weight=0.3)

Processing rows: 100%|██████████| 4897/4897 [10:10<00:00,  8.02row/s, Time Remaining=0min0s]  


Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.83      0.88      2313
           1       0.86      0.94      0.90      2584

    accuracy                           0.89      4897
   macro avg       0.89      0.89      0.89      4897
weighted avg       0.89      0.89      0.89      4897



In [12]:
test(test_raw_df, labels, text_model1, tokenizer1,text_weight=0.7,url_weight=0.4)

Processing rows: 100%|██████████| 4897/4897 [10:19<00:00,  7.90row/s, Time Remaining=0min0s] 

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.82      0.87      2313
           1       0.85      0.95      0.90      2584

    accuracy                           0.88      4897
   macro avg       0.89      0.88      0.88      4897
weighted avg       0.89      0.88      0.88      4897



In [11]:
test(test_raw_df, labels, text_model1, tokenizer1,text_weight=0.8,url_weight=0.2)

Processing rows: 100%|██████████| 4897/4897 [10:02<00:00,  8.13row/s, Time Remaining=0min0s]  


Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.85      0.88      2313
           1       0.88      0.93      0.90      2584

    accuracy                           0.89      4897
   macro avg       0.90      0.89      0.89      4897
weighted avg       0.89      0.89      0.89      4897



In [15]:
test(test_raw_df, labels, text_model1, tokenizer1,text_weight=1,url_weight=0)

Processing rows: 100%|██████████| 4897/4897 [10:07<00:00,  8.07row/s, Time Remaining=0min0s] 

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.88      0.89      2313
           1       0.89      0.91      0.90      2584

    accuracy                           0.89      4897
   macro avg       0.89      0.89      0.89      4897
weighted avg       0.89      0.89      0.89      4897



In [18]:
test(test_raw_df, labels, text_model2, tokenizer2,text_weight=0.8,url_weight=0.2)

Processing rows: 100%|██████████| 4897/4897 [18:14<00:00,  4.48row/s, Time Remaining=0min0s]  

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.88      0.90      2313
           1       0.90      0.94      0.92      2584

    accuracy                           0.91      4897
   macro avg       0.91      0.91      0.91      4897
weighted avg       0.91      0.91      0.91      4897



In [19]:
test(test_raw_df, labels, text_model2, tokenizer2,text_weight=1,url_weight=0)

Processing rows: 100%|██████████| 4897/4897 [18:03<00:00,  4.52row/s, Time Remaining=0min0s]  

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.90      0.91      2313
           1       0.91      0.92      0.92      2584

    accuracy                           0.91      4897
   macro avg       0.91      0.91      0.91      4897
weighted avg       0.91      0.91      0.91      4897

